# 📊 TrendSense — Notebook 1: Data Exploration

**Objective:** Load Walmart & Rossmann datasets, explore distributions, handle missing values, and visualize sales trends.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

import config
from src.data_ingestion import download_walmart_data, fetch_google_trends

# Set style
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

print('✅ Libraries loaded')

## 1. Load Walmart Dataset

In [ ]:
# Load Walmart data
walmart_df = download_walmart_data()

print(f'Shape: {walmart_df.shape}')
print(f'\nColumns: {list(walmart_df.columns)}')
print(f'\nDate range: {walmart_df["Date"].min()} to {walmart_df["Date"].max()}')
print(f'Stores: {walmart_df["Store"].nunique()}')
walmart_df.head(10)

In [ ]:
# Basic statistics
walmart_df.describe()

In [ ]:
# Missing value analysis
missing = walmart_df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values! ✅')

print(f'\nData types:')
print(walmart_df.dtypes)

## 2. Sales Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of Weekly Sales
axes[0].hist(walmart_df['Weekly_Sales'], bins=50, color='#6366F1', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribution of Weekly Sales', fontweight='bold')
axes[0].set_xlabel('Weekly Sales ($)')
axes[0].axvline(walmart_df['Weekly_Sales'].mean(), color='red', linestyle='--', label=f'Mean: ${walmart_df["Weekly_Sales"].mean():,.0f}')
axes[0].legend()

# Box plot by Holiday Flag
walmart_df.boxplot(column='Weekly_Sales', by='Holiday_Flag', ax=axes[1])
axes[1].set_title('Sales by Holiday Flag', fontweight='bold')
axes[1].set_xlabel('Holiday Flag (0=No, 1=Yes)')

# Sales over time
agg = walmart_df.groupby('Date')['Weekly_Sales'].mean()
axes[2].plot(agg.index, agg.values, color='#6366F1', linewidth=1.5)
axes[2].fill_between(agg.index, agg.values, alpha=0.15, color='#6366F1')
axes[2].set_title('Avg Weekly Sales Over Time', fontweight='bold')
axes[2].set_xlabel('Date')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'data_exploration_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distributions plotted')

## 3. Store-Level Analysis

In [ ]:
# Average sales by store
store_avg = walmart_df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False)

fig = px.bar(x=store_avg.index.astype(str), y=store_avg.values,
             labels={'x': 'Store ID', 'y': 'Avg Weekly Sales ($)'},
             title='Average Weekly Sales by Store',
             color=store_avg.values, color_continuous_scale='Viridis')
fig.update_layout(template='plotly_dark', height=400)
fig.show()

print(f'\nHighest performing store: Store {store_avg.index[0]} (${store_avg.iloc[0]:,.0f}/week)')
print(f'Lowest performing store: Store {store_avg.index[-1]} (${store_avg.iloc[-1]:,.0f}/week)')
print(f'Ratio: {store_avg.iloc[0]/store_avg.iloc[-1]:.1f}x')

## 4. Holiday Impact Analysis

In [ ]:
holiday_sales = walmart_df.groupby('Holiday_Flag')['Weekly_Sales'].agg(['mean', 'median', 'std'])
print('Sales Statistics by Holiday Flag:')
print(holiday_sales)

holiday_impact = holiday_sales.loc[1, 'mean'] - holiday_sales.loc[0, 'mean']
holiday_pct = (holiday_impact / holiday_sales.loc[0, 'mean']) * 100

print(f'\n🎯 Holiday Impact: ${holiday_impact:,.0f} ({holiday_pct:+.1f}%) average boost per store per week')

## 5. Correlation Analysis

In [ ]:
numeric_cols = walmart_df.select_dtypes(include=[np.number]).columns
corr = walmart_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Google Trends Data

In [ ]:
# Load Google Trends data
trends_df = fetch_google_trends()

print(f'Shape: {trends_df.shape}')
print(f'Keywords: {[c for c in trends_df.columns if c != "date"]}')
trends_df.head()

In [ ]:
# Plot all trends
keywords = [c for c in trends_df.columns if c != 'date']
fig = px.line(trends_df, x='date', y=keywords,
              title='Google Trends — India Search Interest',
              labels={'value': 'Interest (0-100)', 'date': 'Date'})
fig.update_layout(template='plotly_dark', height=500)
fig.show()

## 7. Key Findings Summary

| Metric | Value |
|--------|-------|
| Dataset Size | See above |
| Missing Values | None |
| Stores | 45 |
| Holiday Impact | Significant positive boost |
| Top Correlations | CPI, Unemployment with Store |

### Next Steps:
- Compute TVI from Google Trends data → **Notebook 02**
- Engineer features for ML models → **Notebook 03**